## 1. Loading the cleaned corpus

In [2]:
import json
from pathlib import Path

corpus_path = Path("C:\\Users\\KlaudiaPoka\\OneDrive - BMW Techworks Romania\\Desktop\\Personal\\AIE\\echochamber-project-team-2\\data\\cleaned\\student_01_youtube_clean.jsonl")

comments = []
with corpus_path.open("r", encoding="utf-8") as f:
    for line in f:
        comments.append(json.loads(line))

len(comments)


74

## 2. Corpus overview

**Total number of comments:** 74


## 3. Main source channels

In [3]:
from collections import Counter

channels = Counter(c["source_channel"] for c in comments if "source_channel" in c)

channels.most_common(10)

[('RecorderRomania', 74)]

## 4. Five example comments

In [4]:
for c in comments[:5]:
    print(c["text"])
    print("---")

Sfat util pentru dobitocii care au votat 36 de ani cu PSD(fostul PCR) acum au opțiunea să voteze cu AUR(PSD2).
---
Discursul lui Farfuridi, de I.Luca Caragiale "Din două una, dați-mi voie: ori să se revizuiască, primesc! Dar să nu se schimbe nimic; ori să nu se revizuiască, primesc! Dar atunci să se schimbe pe aici pe colo, și anume în punctele… esenţiale."
---
Nenorociților , distrugeți o țară !!! Doamne , pedepsește i pe trădători !!
---
Inclusiv dezastrele is provocate ca fie distrus tot ce mai poate produce tara noastra...inca o mină disparută...plus mult ecosistem afectat..compromis
---
se stiu ei , cei care inca isi iau salarii de la stat , in functiile lor caldute .... s-o "demis" guvernul si Dan Dobrea e tot acolo , aproape ca ar fi hilar daca n-ar fi tragic ... si ca el , cati inca ?
---


## 5. 10 comments for prompt testing

In [5]:
test_comments = comments[5:15]

for i, c in enumerate(test_comments, 1):
    print(f"Comment {i}:")
    print(c["text"])
    print("---")

Comment 1:
Amuzant, mina este PA dar salariul celor din conducere merge inainte
---
Comment 2:
Auziți cineva sa ii creadă pe alde Bolojan și Nicușor Dan că lupta împotriva sistemului ...pai ei sunt primi care nu mai vor să audă de așa ceva ... Iar acolo fără să fiu tendențios erwlau ungurii la conducere ...de.asra se.s8 astupa ...
---
Comment 3:
asta e roMANIA … tara unde prostimea voteaza PSD si AUR si apoi se mira ce se intampla !
---
Comment 4:
Este ultima lună în care mai puteți redirecționa 3,5% din impozitul pe venit către Recorder, dacă sunteți salariați. Completați acest formular Durează mai puțin de un minut, iar noi ne vom ocupa să îl încărcăm în spațiul virtual al ANAF.
---
Comment 5:
Am ajuns sa ne uitam la aceste drame ca la un reality show. Inca un semnal de alarma pentru o societate profund amortita. Multumiri Recorder pentru ca sunteti printre putinii care se mai obosesc sa sublineze aceste aspecte.
---
Comment 6:
Ce rău îmi pare de saracii oameni din zona aceea. E irea

## First prompt

> Identify the following for the comment below:
> - Target
> - Sentiment
> - Stance toward the target
> - Topic
> - Ambiguity or interpretation problems

In [6]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii din social media.
Răspunzi concis, clar și te bazezi doar pe informațiile din comentariu.
Dacă ceva este ambiguu sau neclar, menționează explicit acest lucru.
Nu inventa informații.
"""

PROMPT_TEMPLATE = """
Analizează următorul comentariu:

"{comment}"

Răspunde structurat, câte un rând pentru fiecare:
Țintă (target):
Sentiment (pozitiv / negativ / neutru):
Poziționare față de țintă (susținere / critică / neutră / neclară):
Topic:
Ambiguități sau probleme de interpretare:
"""

In [7]:
from openai import OpenAI
import google.generativeai as genai
import os

def ask(provider, model, prompt, system="", temperature=0):
    if provider == "gemini":
        genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
        m = genai.GenerativeModel(model)
        r = m.generate_content(
            system + "\n" + prompt,
            generation_config={"temperature": temperature}
        )
        return r.text

    elif provider == "openrouter":
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY")
        )
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature
        )
        return r.choices[0].message.content

    elif provider == "groq":
        client = OpenAI(
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY")
        )
        r = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature
        )
        return r.choices[0].message.content

    else:
        raise ValueError(f"Unknown provider: {provider}")

c:\Users\KlaudiaPoka\OneDrive - BMW Techworks Romania\Desktop\Personal\AIE\echochamber-project-team-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\KlaudiaPoka\AppData\Local\Temp\ipykernel_4000\3367793680.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [15]:
import os
print("GOOGLE_API_KEY value:", os.getenv("GOOGLE_API_KEY"))

GOOGLE_API_KEY value: None


In [24]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Find project root (walk up until .env is found)
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

print("Project root detected as:", ROOT)

load_dotenv(ROOT / ".env", override=True)

print("GOOGLE_API_KEY loaded:", os.getenv("GOOGLE_API_KEY") is not None)

Project root detected as: c:\Users\KlaudiaPoka\OneDrive - BMW Techworks Romania\Desktop\Personal\AIE\echochamber-project-team-2
GOOGLE_API_KEY loaded: True


In [27]:
import google.generativeai as genai

for m in genai.list_models():
    print(m.name, m.supported_generation_methods)

models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite-001 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-4-26b-a4b-it ['generateContent', 'countTokens']
models/gemma-4-31b-it ['generateContent', 'countTokens']
models/gemini-flash-latest ['generateContent', 'countTokens', 'cre

In [28]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

m = genai.GenerativeModel("gemini-flash-latest")
response = m.generate_content("Spune un salut scurt.")
print(response.text)

Salut!


## Test with Gemini first 6 comments 

In [ ]:
provider = "gemini" 
model = "gemini-flash-latest" 

for i, c in enumerate(test_comments, 1):
    print("=" * 80)
    print(f"Comentariu {i} – TEXT ORIGINAL:")
    print(c["text"])
    print("-" * 80)
    print("ANALIZĂ:")
    print(ask(
        provider=provider,
        model=model,
        system=SYSTEM,
        prompt=PROMPT_TEMPLATE.format(comment=c["text"]),
        temperature=0
    ))
    print("\n")

Comentariu 1 – TEXT ORIGINAL:
Amuzant, mina este PA dar salariul celor din conducere merge inainte
--------------------------------------------------------------------------------
ANALIZĂ:
Țintă (target): Conducerea (managementul) și politica salarială a acesteia.
Sentiment (pozitiv / negativ / neutru): Negativ.
Poziționare față de țintă (susținere / critică / neutră / neclară): Critică.
Topic: Discrepanța dintre închiderea/eșecul minei și continuitatea veniturilor conducerii.
Ambiguități sau probleme de interpretare: Abrevierea „PA” este informală (probabil semnificând „adio”/închisă); nu este specificată identitatea minei.


Comentariu 2 – TEXT ORIGINAL:
Auziți cineva sa ii creadă pe alde Bolojan și Nicușor Dan că lupta împotriva sistemului ...pai ei sunt primi care nu mai vor să audă de așa ceva ... Iar acolo fără să fiu tendențios erwlau ungurii la conducere ...de.asra se.s8 astupa ...
--------------------------------------------------------------------------------
ANALIZĂ:
Țintă (

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash
Please retry in 40.15106005s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 40
}
]

In [31]:
last_4_comments = test_comments[-4:]
len(last_4_comments)

4

## Test with grok the comments because gemini ran out of free requests


In [35]:
provider = "groq"
model = "llama-3.1-8b-instant"

for i, c in enumerate(test_comments, 1):
    print("=" * 80)
    print(f"Comentariu {i} – TEXT ORIGINAL:")
    print(c["text"])
    print("-" * 80)
    print("ANALIZĂ:")
    print(ask(
        provider=provider,
        model=model,
        system=SYSTEM,
        prompt=PROMPT_TEMPLATE.format(comment=c["text"]),
        temperature=0
    ))
    print("\n")

Comentariu 1 – TEXT ORIGINAL:
Amuzant, mina este PA dar salariul celor din conducere merge inainte
--------------------------------------------------------------------------------
ANALIZĂ:
Aici este analiza comentariului:

**Țintă (target):**
Conducerea unei companii (în special cei care primesc salarii mari).

**Sentiment:**
Negativ.

**Poziționare față de țintă:**
Criticismă.

**Topic:**
Salariile conducătorilor unei companii în raport cu salariile angajaților.

**Ambiguități sau probleme de interpretare:**
Niciuna, comentariul este clar și direct.


Comentariu 2 – TEXT ORIGINAL:
Auziți cineva sa ii creadă pe alde Bolojan și Nicușor Dan că lupta împotriva sistemului ...pai ei sunt primi care nu mai vor să audă de așa ceva ... Iar acolo fără să fiu tendențios erwlau ungurii la conducere ...de.asra se.s8 astupa ...
--------------------------------------------------------------------------------
ANALIZĂ:
**Țintă (target):**
- Bolojan
- Nicușor Dan

**Sentiment (pozitiv / negativ / neutr

## Reflection

#### Where did the prompt work well?

 - Where the target was obvious and the tone was clear. Grok failed to write correctly or to identify the target when it was not so specific. Gemini did better. 

#### Where did it fail?

 - To identify the target precicely in some cases

#### Did it confuse sentiment with stance?

 - Sometimes yes. Critical is not always the answer...maybe it it frustrartion or or something else.

#### Did sarcasm, ambiguity, or multiple targets create problems?

 - Yes, again at target identification or sentiment recognition.

#### What would I change in the next version of the prompt?

 - Require the model to mark cases as "target unclear" and to distinguish emotional reaction from evaluative stance, especially in comments expressing frustration, anger, or resignation


